# Lab 14 · Agent frameworks: graphs with LangGraph

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Generate, check, and repair an event contract
**Start with a problem:** A hand-written workflow can become difficult to follow when a failed contract check needs a retry.

**In this lab:** Represent the generation, check, and bounded repair as graph nodes and edges; still review the proposed contract.

**New:** LangGraph makes workflow state, nodes, and conditional edges explicit. Our graph proposes a change to ShopEasy's `OrderPlaced` event, checks compatibility, and allows one repair.

The rule for this exercise is **additive contract evolution**: keep the existing `orderId` and `total` fields with their types. Add a new optional field rather than renaming `total` to `amount`. This notebook uses a tiny type-map notation, not a complete JSON Schema validator or production schema registry.

**At a glance**

![Lab 14 at a glance: START (OrderPlaced contract), then choose (node), then check (keep old names + types), then route, then END. Then back to choose: retry · at most 2 attempts.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/14.svg)

## Think about the example

Payment reads the total field in an OrderPlaced event. Renaming or removing that field breaks its existing contract. A new optional field can be added without requiring existing consumers to use it.

**Concepts:** Additive event changes; Published contract; Bounded generator–critic repair.

Use the scenario above to consider these questions before running the code:

1. Why does renaming total break a consumer?
2. Why should a new event field be optional for existing consumers?

## The idea in context

A graph names workflow steps and conditional edges. Generation and checking remain distinct so failed proposals receive explicit feedback.

An additive event change protects consumers; a framework does not make a proposal compatible automatically.

| Key term | Meaning |
| --- | --- |
| **State** | The data that all steps of a workflow read and write. |
| **Node** | One step of the graph, written as a function of the state. |
| **Edge** | A connection that sets which node runs next. |
| **Conditional edge** | An edge that picks the next node from the state, which is how a graph loops. |

## Pattern · Generator–Critic Loop

A generator proposes; a separate check returns concrete problems; a conditional edge revises or stops. A two-attempt cap prevents endless churn. The critic here is deterministic contract checking, not a second model.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit', 'langgraph']):
    %pip install -q --upgrade "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Step 1 · Describe state and two changes

A `TypedDict` documents fields shared by nodes; it does not validate model output at runtime. `CURRENT` is the published event contract. New fields must use our illustrative `optional string` type so existing consumers are not required to supply them.

`State` names the information passed between nodes. `total=False` allows a run to start with only the project; later nodes add the remaining fields.

In [ ]:
from typing import TypedDict

class State(TypedDict, total=False):
    project: dict
    fields: dict
    attempts: int
    passed: bool
    feedback: str

CURRENT = {"orderId": "string", "total": "number"}
projects = [{"name": "Add coupon", "new_field": "couponCode"},
            {"name": "Add campaign", "new_field": "campaignId"}]

## Step 2 · Write generation and checking nodes

`choose` returns only updated state fields. Each attempt calls the selected model with the latest feedback. `check` validates the JSON object's shape, existing names and types, and the requested optional addition. This compact check is specific to our controlled example.

The `choose` node returns the proposed fields and an updated attempt count. The `check` node verifies that existing fields keep their names and types.

In [ ]:
import json

def choose(state):
    project = state["project"]
    attempt = state.get("attempts", 0)
    prompt = (f"Existing event: {CURRENT}. Add {project['new_field']} without breaking consumers. Keep existing names/types. "
              f"For the new field, use the exact type-label text 'optional string', not 'string?'. Reply with only a JSON object, no fences. {state.get('feedback', '')}")
    return {"fields": json.loads(ask(prompt)), "attempts": attempt + 1}

def check(state):
    fields = state["fields"]
    new = state["project"]["new_field"]
    ok = isinstance(fields, dict) and all(fields.get(k) == v for k, v in CURRENT.items())
    ok = ok and fields.get(new) == "optional string" and all(k in CURRENT or v == "optional string" for k, v in fields.items())
    return {"passed": ok, "feedback": "" if ok else f"Keep {CURRENT}; the value for {new} must be exactly 'optional string'. Do not rename total."}

print(check({"project": projects[0], "fields": {"orderId": "string", "amount": "number"}}))

## Step 3 · Connect the graph

The sequence is `START → choose → check`. A failed check returns to `choose` unless two attempts have already run. Invalid JSON is rejected explicitly by `json.loads`; a non-object JSON reply reaches the checker and fails safely.

A conditional edge asks `route` where to go after the check. It retries once at most; a second failure ends for human review.

In [ ]:
from langgraph.graph import StateGraph, START, END

def route(state):
    return "done" if state["passed"] or state["attempts"] >= 2 else "retry"

graph = StateGraph(State)
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_edge(START, "choose")
graph.add_edge("choose", "check")
graph.add_conditional_edges("check", route, {"retry": "choose", "done": END})
app = graph.compile()
for edge in app.get_graph().edges:
    print(edge.source, "->", edge.target)

## Step 4 · Run both changes

The graph asks for each change and checks the response. If a proposal breaks the contract, it requests a repair; model outputs and attempt counts vary.

In [ ]:
results = [app.invoke({"project": project}) for project in projects]
for r in results:
    print(r["project"]["name"], r["fields"], "attempts:", r["attempts"], "passed:", r["passed"])
print("Generation attempts:", sum(r["attempts"] for r in results))
assert all(r["passed"] for r in results), "A contract still breaks after two attempts; stop and review."

## What you should see

The bad contract fails the direct check. For each project, the graph asks the model for a contract and retries at most once if needed. A valid contract retains `orderId: string` and `total: number`, plus the optional new field; attempts vary by response.

## Try it

1. Test `check` with a contract that renames `total`. What feedback would the graph send to the model?
2. Send a non-object JSON reply such as `[]`. Which check rejects it?
3. Change the `total` type from number to string. Why is that breaking even though the field still exists?
4. **Use it in your council.** Make proposal generation and contract review two separate graph nodes. Which edge should escalate an unresolved contract change to a human?

## Takeaway

A framework organizes state and retries; the domain checker defines correctness. Additive contracts reduce release coupling, but real compatibility also requires agreed semantics, defaults, versioning, and consumer tests.